# recs_031 — Retune the v2a ranker's alpha/w_meta for RAG retrieval pools

`alpha=0.2` / `w_meta=0.1` were tuned in `recs_013`/`recs_020` against `two_tower_v1` pools --
never re-checked against Stage 3's RAG retrieval (`rag_chunk_v1_vector_blend_query`), which is
what's actually shipped today (`RAGRecommender`). We already found that combination underperforms
the old `two_tower_v1` + v2a stack end-to-end (0.089 vs 0.095 NDCG@10 overall on val), and that
v2a is worse than *bare* RAG retrieval order at the low-popularity deciles (D1-D3) -- exactly
where a genuinely content-driven reranker should help most, not hurt.

This notebook sweeps `alpha` (holding `w_meta=0.1` fixed, matching the shipped default) on the
**train** RAG pool (`rag_chunk_v1_vector_blend_query.parquet`, 51,691 examples -- no val tuning),
with a guardrail: a candidate can't score worse than bare RAG retrieval order at D1-D3. Whatever
passes the guardrail with the best Slice A NDCG gets validated once on held-out val.

Reuses the actual production rerank functions (`score_v2a_embed_query_logpop_blend`,
`heuristic_ranker.py`/`v2a_metadata_ranker.py`) rather than reimplementing the blend formula --
same lesson as the Chroma batching work: don't duplicate logic that already exists.


In [2]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd

from steam_review_ml.evaluation.example_cohort import load_retrieval_pool_rows, load_retrieval_pools_jsonl
from steam_review_ml.evaluation.retrieval_offline_eval import (
    _example_popularity_segments,
    _rank_rows,
    load_ranking_catalog_context,
    ndcg_at_k,
)
from steam_review_ml.evaluation.v2a_metadata_ranker import (
    DEFAULT_V2A_EMBED_W_META,
    score_v2a_embed_query_logpop_blend,
)

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())

POOL_METHOD = "rag_chunk_v1_vector_blend_query"
K_FINAL = 10
MIN_REVIEW_CHARS = 30
ARTIFACT_DIR = REPO_ROOT / "artifacts/recs"
GUARDRAIL_DECILES = {"D1", "D2", "D3"}

TRAIN_POOLS_PARQUET = REPO_ROOT / "artifacts/recs/ranker_pools/train_ranker_v1/rag_chunk_v1_vector_blend_query.parquet"
VAL_JSONL = REPO_ROOT / "artifacts/recs/offline_eval/runs/rag_v3/eval_offline_examples.jsonl"


/home/ryanr/miniconda3/envs/tf_condaforge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-27 08:24:27.950325: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-27 08:24:27.963148: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787833467.977646  153124 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787833467.983557  153124 cuda_bl

## Load train pool + catalog context (popularity deciles use the same `pd.qcut` bucketing the eval jobs already use)

In [3]:
train_pools = load_retrieval_pool_rows(TRAIN_POOLS_PARQUET)

catalog = load_ranking_catalog_context(
    repo_root=REPO_ROOT,
    min_review_chars=MIN_REVIEW_CHARS,
    artifact_dir=ARTIFACT_DIR,
)
app_ids = catalog.app_ids
app_to_row = catalog.app_to_row
pop_row = catalog.pop_row

examples_for_decile = [
    {"validation_positive_app_ids": json.loads(row["validation_positive_app_ids_json"])}
    for row in sorted(train_pools, key=lambda r: int(r["ex_idx"]))
]
ex_pop_map = _example_popularity_segments(examples=examples_for_decile, app_ids=app_ids, pop_row=pop_row)
decile_by_ex_idx = ex_pop_map.set_index("ex_idx")["pos_pop_decile"]

print(f"train pools: {len(train_pools):,}  catalog apps: {len(app_ids):,}")


train pools: 51,691  catalog apps: 315


## Metric helpers

In [4]:
def _pool_ndcg(row: dict, scores: np.ndarray, pool_apps: list[int]) -> float | None:
    positives = {int(x) for x in json.loads(row["validation_positive_app_ids_json"])}
    if not positives:
        return None
    order = _rank_rows(scores)[:K_FINAL]
    ranked_rows = np.asarray([app_to_row[pool_apps[i]] for i in order])
    return ndcg_at_k(ranked_rows, positives, K_FINAL, app_ids)


def mean_ndcg(
    pools: list[dict],
    rerank_fn: Callable[..., np.ndarray] | None,
    params: dict[str, Any] | None = None,
    *,
    slice_a_only: bool = False,
    deciles: set[str] | None = None,
) -> float:
    """Mean NDCG@10. ``rerank_fn=None`` -> bare retrieval order (no rerank at all)."""
    vals: list[float] = []
    for row in pools:
        if slice_a_only and int(row["n_eval_targets"]) < 2:
            continue
        if deciles is not None and decile_by_ex_idx.get(int(row["ex_idx"])) not in deciles:
            continue
        pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
        ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
        if rerank_fn is None:
            scores = np.asarray(ret_sc)
        else:
            scores = rerank_fn(
                pool_apps, ret_sc,
                pop_row=pop_row, app_to_row=app_to_row, query_app_id=int(row["query_app_id"]),
                **(params or {}),
            )
        v = _pool_ndcg(row, scores, pool_apps)
        if v is not None:
            vals.append(v)
    return float(np.mean(vals)) if vals else float("nan")


## Guardrail floor: bare RAG retrieval order at D1-D3 (no rerank at all)

In [5]:
bare_guardrail_ndcg = mean_ndcg(train_pools, rerank_fn=None, deciles=GUARDRAIL_DECILES)
print(f"Bare RAG retrieval order, D1-D3 mean NDCG@10 (guardrail floor): {bare_guardrail_ndcg:.4f}")


Bare RAG retrieval order, D1-D3 mean NDCG@10 (guardrail floor): 0.0772


## Sweep alpha (w_meta fixed at the shipped default, 0.1)

Uses `score_v2a_embed_query_logpop_blend` for every point (including small alpha) rather than
switching functions -- it already collapses to pure D1 behavior internally when `w_meta<=0`, so
one code path covers both without a branch that could drift out of sync.

This computes taxonomy-similarity lookups for ~52k examples x len(ALPHA_GRID) -- may take a
few minutes.


In [6]:
ALPHA_GRID = [0.2, 0.35, 0.5, 0.65, 0.8]
W_META = DEFAULT_V2A_EMBED_W_META  # 0.1, matches the shipped default -- not swept here

results = []
for alpha in ALPHA_GRID:
    params = {"alpha": alpha, "w_meta": W_META}
    overall = mean_ndcg(train_pools, score_v2a_embed_query_logpop_blend, params)
    slice_a = mean_ndcg(train_pools, score_v2a_embed_query_logpop_blend, params, slice_a_only=True)
    guardrail = mean_ndcg(train_pools, score_v2a_embed_query_logpop_blend, params, deciles=GUARDRAIL_DECILES)
    results.append(
        {
            "alpha": alpha,
            "w_meta": W_META,
            "ndcg_overall": overall,
            "ndcg_slice_a": slice_a,
            "ndcg_d1_d3": guardrail,
            "passes_guardrail": guardrail >= bare_guardrail_ndcg,
        }
    )

results_df = pd.DataFrame(results).sort_values("ndcg_slice_a", ascending=False).reset_index(drop=True)
results_df


,alpha,w_meta,ndcg_overall,ndcg_slice_a,ndcg_d1_d3,passes_guardrail
0,0.20,0.1,0.160186,0.168809,0.050380,False
1,0.35,0.1,0.141948,0.150398,0.073585,False
2,0.50,0.1,0.116814,0.124640,0.085374,True
3,0.65,0.1,0.096860,0.104000,0.087095,True
4,0.80,0.1,0.082464,0.088852,0.084319,True


## Follow-up: sweep w_meta instead (alpha fixed at the shipped default)

The alpha sweep above assumed w_meta's effect is negligible -- but that was only ever measured
on the old two_tower_v1 pools (D1 0.093 -> v2a 0.095, a +0.002 nudge). Never checked whether
that holds for RAG's different score distribution. Holding alpha at the shipped default and
sweeping w_meta tests that assumption directly instead of carrying it over unchecked.


In [7]:
ALPHA = 0.2  # shipped default; the alpha sweep above found no alpha wins outright, so hold it fixed
W_META_GRID = [0.0, 0.1, 0.2, 0.3]

w_meta_results = []
for w_meta in W_META_GRID:
    params = {"alpha": ALPHA, "w_meta": w_meta}
    overall = mean_ndcg(train_pools, score_v2a_embed_query_logpop_blend, params)
    slice_a = mean_ndcg(train_pools, score_v2a_embed_query_logpop_blend, params, slice_a_only=True)
    guardrail = mean_ndcg(train_pools, score_v2a_embed_query_logpop_blend, params, deciles=GUARDRAIL_DECILES)
    w_meta_results.append(
        {
            "alpha": ALPHA,
            "w_meta": w_meta,
            "ndcg_overall": overall,
            "ndcg_slice_a": slice_a,
            "ndcg_d1_d3": guardrail,
        }
    )

w_meta_results_df = pd.DataFrame(w_meta_results).sort_values("ndcg_slice_a", ascending=False).reset_index(drop=True)
w_meta_results_df


,alpha,w_meta,ndcg_overall,ndcg_slice_a,ndcg_d1_d3
0,0.2,0.1,0.160186,0.168809,0.050380
1,0.2,0.0,0.159928,0.168512,0.046704
2,0.2,0.2,0.157624,0.166663,0.056357
3,0.2,0.3,0.153389,0.162696,0.062308


## Pick the winner: best Slice A NDCG *among* candidates that pass the guardrail

In [8]:
winners = results_df[results_df["passes_guardrail"]].sort_values("ndcg_slice_a", ascending=False)
if winners.empty:
    print("No candidate passed the D1-D3 guardrail -- widen ALPHA_GRID or reconsider the guardrail bar.")
else:
    best = winners.iloc[0]
    print(f"Winner: alpha={best['alpha']}, w_meta={best['w_meta']}")
    print(f"  train ndcg_overall={best['ndcg_overall']:.4f}  ndcg_slice_a={best['ndcg_slice_a']:.4f}")
winners


Winner: alpha=0.5, w_meta=0.1
  train ndcg_overall=0.1168  ndcg_slice_a=0.1246


,alpha,w_meta,ndcg_overall,ndcg_slice_a,ndcg_d1_d3,passes_guardrail
2,0.50,0.1,0.116814,0.124640,0.085374,True
3,0.65,0.1,0.096860,0.104000,0.087095,True
4,0.80,0.1,0.082464,0.088852,0.084319,True


## Validate the winner on held-out val (no tuning here -- fixed params from above)

Compare against what we already measured this session for the current default and the old
shipped stack.


In [9]:
val_pools = load_retrieval_pools_jsonl(VAL_JSONL, method=POOL_METHOD)

best_params = {"alpha": float(best["alpha"]), "w_meta": float(best["w_meta"])}
val_overall = mean_ndcg(val_pools, score_v2a_embed_query_logpop_blend, best_params)
val_slice_a = mean_ndcg(val_pools, score_v2a_embed_query_logpop_blend, best_params, slice_a_only=True)

print(f"Winner (alpha={best_params['alpha']}, w_meta={best_params['w_meta']}) on held-out val:")
print(f"  NDCG@10 overall={val_overall:.4f}  slice_a={val_slice_a:.4f}")
print()
print("For comparison (measured earlier this session, artifacts/recs/offline_eval/runs/rag_v3_ranking/):")
print("  old shipped stack   (two_tower_v1 + v2a, alpha=0.2/w_meta=0.1): overall=0.0953  slice_a=0.0703")
print("  current RAG default (alpha=0.2/w_meta=0.1, unchanged):         overall=0.0887  slice_a=0.0605")
print("  popularity_train baseline:                                     overall=0.0731  slice_a=0.0354")


Winner (alpha=0.5, w_meta=0.1) on held-out val:
  NDCG@10 overall=0.0754  slice_a=0.0613

For comparison (measured earlier this session, artifacts/recs/offline_eval/runs/rag_v3_ranking/):
  old shipped stack   (two_tower_v1 + v2a, alpha=0.2/w_meta=0.1): overall=0.0953  slice_a=0.0703
  current RAG default (alpha=0.2/w_meta=0.1, unchanged):         overall=0.0887  slice_a=0.0605
  popularity_train baseline:                                     overall=0.0731  slice_a=0.0354


## Next step if this actually wins

Only promote into `configs/recs_serve.json` if the winner both clears the old shipped stack's
val numbers *and* still passes the D1-D3 guardrail on val (recompute via
`recs_job_eval_ranking.py` with a config like `rag_v3_ranking`'s, swapping in the winning
alpha/w_meta) -- this notebook's val check above is a quick gate, not the full decile audit.
